In [ ]:
import os
import re
from typing import List, Dict, Tuple, Optional
import tempfile
import shutil
from collections import Counter
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Suppress warnings that might clutter the notebook output
warnings.filterwarnings('ignore', category=UserWarning)

# ==============================================================================
# USER CONFIGURABLE PARAMETERS
# ==============================================================================

# --- File Paths and Patterns ---
# Root directory for finding input files.
SEARCH_ROOT_DIR = "."

# The input file pattern generated by the ΔSASA Scoring script.
REPLICA_FILENAME_TEMPLATE = "deltaSASA_repl_{repl}_scored.xlsx"

# The output filename pattern for the consensus results.
OUTPUT_XLSX_PATTERN = "consensus_score_{score_threshold}.xlsx"

# --- Analysis Parameters ---
DEFAULT_CHAINS = ["A", "B"]
SCORE_THRESHOLD_DEFAULT = 0.7

# Input sheet names from the scored XLSX file.
SHEET_MEAN_DATA = "mean_delta_scored"
SHEET_MEDIAN_DATA = "median_delta_scored"

# Required columns in the input DataFrame after scoring.
COLS_REQUIRED = ["chain", "residue_pdb_resseq", "resname", "weighted_score_total"]

# ==============================================================================
# HELPER FUNCTIONS (Data Loading and Manipulation)
# ==============================================================================

def find_available_replicas(analysis_dir: str, template: str) -> List[int]:
    """
    Searches the analysis directory for files matching the template and extracts
    the replica numbers dynamically.
    """
    analysis_path = Path(analysis_dir).resolve()
    # Create a glob pattern to match any number in the {repl} position
    glob_pattern = template.replace("{repl}", "[0-9]*") 
    
    found_replicas = set()
    # Define a regex to extract the number accurately
    regex_pattern = re.compile(re.escape(template).replace(r"\{repl\}", r"(\d+)"))
    
    for f in analysis_path.glob(glob_pattern):
        match = regex_pattern.search(str(f.name))
        if match:
            found_replicas.add(int(match.group(1)))
            
    return sorted(list(found_replicas))


def load_replica_file(filepath: Path, repl_num: int, sheet_name: str, required_columns: List[str]) -> Optional[pd.DataFrame]:
    """
    Loads a specific sheet from an Excel replica file and validates the presence
    of required columns.
    """
    file_identifier = f"Replica [{repl_num}]"
    if not filepath.exists():
        print(f"{file_identifier} WARNING ⚠️ - File not found: {filepath.name}")
        return None
    try:
        df = pd.read_excel(filepath, sheet_name=sheet_name, engine=None)
    except Exception as e:
        print(f"{file_identifier} ERROR ❌ - Could not read sheet '{sheet_name}': {e}")
        return None

    df.columns = [c.strip() if isinstance(c, str) else c for c in df.columns]
    
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        print(f"{file_identifier} ERROR ❌ - Missing required columns in sheet '{sheet_name}': {missing}")
        return None

    df = df.copy()
    df["chain"] = df["chain"].astype(str)
    df["resname"] = df["resname"].astype(str)
    
    # Ensure the PDB sequence number is treated as a string
    if "residue_pdb_resseq" in df.columns:
        df["residue_pdb_resseq"] = df["residue_pdb_resseq"].apply(lambda v: v if pd.isna(v) else str(v))

    print(f"{file_identifier} INFO - Sheet '{sheet_name}' loaded successfully.")

    return df

def make_residue_key(row: pd.Series) -> str:
    """
    Generates a unique string key for a residue: 'Chain|ResidueName|ResiduePDBSeq'.
    """
    return f"{str(row['chain'])}|{str(row['resname'])}|{str(row['residue_pdb_resseq'])}"

def parse_residue_key(key: str) -> Tuple[str, str, str]:
    """
    Parses a residue key back into its components (Chain, ResiduePDBSeq, ResName).
    """
    parts = key.split("|")
    if len(parts) != 3: return "", "", ""
    return parts[0], parts[2], parts[1] 

def build_parallel_sheet(replica_dfs: Dict[int, Optional[pd.DataFrame]], chains: List[str]) -> pd.DataFrame:
    """
    Build the parallel (side-by-side) table for replicas based on the required columns.
    """
    per_replica_chain_rows: Dict[int, Dict[str, pd.DataFrame]] = {}
    for r, df in replica_dfs.items():
        per_replica_chain_rows[r] = {}
        subset_cols = [c for c in ["chain", "residue_pdb_resseq", "resname", "weighted_score_total"] if c in COLS_REQUIRED]
        
        for chain in chains:
            if df is None:
                per_replica_chain_rows[r][chain] = pd.DataFrame(columns=subset_cols)
            else:
                sel = df[df["chain"] == chain][subset_cols].copy()
                per_replica_chain_rows[r][chain] = sel.reset_index(drop=True)

    rows_per_chain = {}
    for chain in chains:
        rows_per_chain[chain] = max((len(per_replica_chain_rows[r][chain]) for r in per_replica_chain_rows.keys()), default=0)

    blocks = []
    for chain in chains:
        nrows = rows_per_chain[chain]
        if nrows == 0:
            continue
        
        block = pd.DataFrame(index=range(nrows))
        
        for r in sorted(per_replica_chain_rows.keys()):
            sub = per_replica_chain_rows[r][chain]
            sub_reindexed = sub.reindex(range(nrows)).reset_index(drop=True)
            
            block[f"r{r}_chain"] = sub_reindexed["chain"]
            block[f"r{r}_resseq"] = sub_reindexed["residue_pdb_resseq"]
            block[f"r{r}_resname"] = sub_reindexed["resname"]
            block[f"r{r}_score"] = sub_reindexed["weighted_score_total"]
            
        blocks.append(block)
    
    if not blocks:
        return pd.DataFrame()
        
    result = pd.concat(blocks, ignore_index=True)
    return result

def filter_residues_by_score(df: Optional[pd.DataFrame], chain: str, score_threshold: float) -> List[str]:
    """
    Select residues for a given replica & chain using its per-replica score column.
    """
    if df is None or df.empty:
        return []
    sel = df[(df["chain"] == chain) & (pd.to_numeric(df["weighted_score_total"], errors="coerce") >= score_threshold)]
    return [make_residue_key(row) for _, row in sel.iterrows()]

def order_union_preserve_first_appearance(replica_dfs: Dict[int, Optional[pd.DataFrame]], chains: List[str]) -> List[str]:
    """
    Creates a master ordered list of unique residue keys across all replicas.
    """
    seen = set()
    union_ordered = []
    for chain in chains:
        for r in sorted(replica_dfs.keys()):
            df = replica_dfs.get(r)
            if df is None:
                continue
            for _, row in df.iterrows():
                if str(row["chain"]) != chain:
                    continue
                key = make_residue_key(row)
                if key not in seen:
                    seen.add(key)
                    union_ordered.append(key)
    return union_ordered

# ==============================================================================
# PLOTTING FUNCTIONS
# ==============================================================================

def plot_frequency_distribution(residue_keys: List[str], replicas_key_lists: Dict[int, List[str]], title: str) -> plt.Figure:
    """
    Bar plot: residues (x) vs number of replicas where they pass threshold (y).
    """
    counts, filtered_keys = [], []
    for k in residue_keys:
        c = sum(1 for lst in replicas_key_lists.values() if k in lst)
        if c > 0:
            counts.append(c)
            filtered_keys.append(k)

    fig, ax = plt.subplots(figsize=(max(6, 0.5 * len(filtered_keys)), 4))
    xs = list(range(len(filtered_keys)))
    ax.bar(xs, counts)
    
    labels = [f"{parse_residue_key(k)[1]}{parse_residue_key(k)[2]}" for k in filtered_keys]
    
    ax.set_xticks(xs)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_ylabel("Number of replicas")

    available_repls = max(1, len(replicas_key_lists))
    max_count = max(counts) if counts else 0
    y_max = max(available_repls, max_count, 1)
    ax.set_ylim(0, y_max + 0.5)
    ax.set_yticks(range(0, int(y_max) + 1))
    ax.set_title(title)
    fig.tight_layout()
    return fig

def plot_overlap_matrix(residue_keys: List[str], replicas_key_lists: Dict[int, List[str]], title: str) -> plt.Figure:
    """
    Matrix plot showing which residues were selected in which replica.
    """
    filtered_keys = []
    for k in residue_keys:
        c = sum(1 for lst in replicas_key_lists.values() if k in lst)
        if c > 0:
            filtered_keys.append(k)

    n_res = len(filtered_keys)
    repls = sorted(replicas_key_lists.keys())
    n_repl = len(repls)
    matrix = np.zeros((n_repl, n_res), dtype=int)
    
    for i, r in enumerate(repls):
        present = set(replicas_key_lists.get(r, []) or [])
        for j, k in enumerate(filtered_keys):
            matrix[i, j] = 1 if k in present else 0

    fig, ax = plt.subplots(figsize=(max(6, 0.5 * n_res), max(3, 0.6 * n_repl)))
    
    xs, ys, sizes, facecolors, edgecolors = [], [], [], [], []
    for i in range(n_repl):
        for j in range(n_res):
            xs.append(j)
            ys.append(n_repl - 1 - i)
            is_on = matrix[i, j] == 1
            sizes.append(200)
            facecolors.append("black" if is_on else "white")
            edgecolors.append("black")
            
    ax.scatter(xs, ys, s=sizes, c=facecolors, edgecolors=edgecolors)
    
    labels = [f"{parse_residue_key(k)[1]}{parse_residue_key(k)[2]}" for k in filtered_keys]
    ax.set_xticks(range(n_res))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    
    ax.set_yticks(range(n_repl))
    ax.set_yticklabels([f"repl{r}" for r in repls][::-1])
    
    ax.set_title(title)
    ax.set_xlim(-0.5, max(n_res - 0.5, 0.5))
    ax.set_ylim(-0.5, n_repl - 0.5)
    fig.tight_layout()
    return fig


def plot_per_replica_totals(replicas_dfs: Dict[int, Optional[pd.DataFrame]],
                            chains: List[str],
                            title: str) -> plt.Figure:
    """
    Horizontal bar chart showing the total number of flagged residues per replica and chain.
    """
    repls = sorted(replicas_dfs.keys())
    counts, labels = [], []
    
    for r in repls:
        df = replicas_dfs.get(r)
        for chain in chains:
            if df is None or df.empty:
                counts.append(0)
            else:
                sel = df[df["chain"] == chain]
                counts.append(len(sel))
            labels.append(f"repl{r} chain{chain}")
            
    fig, ax = plt.subplots(figsize=(8, max(2, 0.5 * len(labels))))
    y_pos = np.arange(len(labels))
    ax.barh(y_pos, counts)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels)
    ax.set_xlabel("Number of residues")
    ax.set_title(title)
    fig.tight_layout()
    return fig

# ==============================================================================
# EXCEL OUTPUT WRITING
# ==============================================================================

def write_excel_output(output_path: Path,
                       mean_table: pd.DataFrame,
                       median_table: pd.DataFrame,
                       summary_content: Dict,
                       images: List[Tuple[str, str]],
                       engine: str = "xlsxwriter") -> None:
    """
    Writes the final aggregated and consensus results to Excel.
    """
    os.makedirs(output_path.parent, exist_ok=True)
    writer = pd.ExcelWriter(output_path, engine=engine)
    
    try:
        summary_content["freq_table"].to_excel(writer, sheet_name="consensus_frequency", index=False)
        mean_table.to_excel(writer, sheet_name="mean_list", index=False)
        median_table.to_excel(writer, sheet_name="median_list", index=False)

        workbook = writer.book
        summary_sheet_name = "summary"
        worksheet = workbook.add_worksheet(summary_sheet_name)
        writer.sheets[summary_sheet_name] = worksheet

        header_format = workbook.add_format({'bold': True, 'bg_color': '#DDEBF7', 'border': 1})
        title_format = workbook.add_format({'bold': True, 'font_size': 12})
        
        start_row = 0
        
        available_repls = summary_content.get('available_replicas', [])
        score_threshold = summary_content.get('score_threshold')
        soft_threshold_val = summary_content.get('soft_threshold_val', 2) 
        
        info_text = f"Available replicas: {','.join(map(str, available_repls))}    threshold: {score_threshold}"
        
        worksheet.write(start_row, 0, info_text, title_format)
        start_row += 2

        consensus = summary_content.get("consensus", {})
        soft_desc = f"SOFT consensus (present in >={soft_threshold_val} but <{len(available_repls)} replicas)"
        
        blocks = [
            ("mean_hard", "Mean - HARD consensus (present in ALL available replicas)"),
            ("mean_soft", f"Mean - {soft_desc}"), 
            ("median_hard", "Median - HARD consensus (present in ALL available replicas)"),
            ("median_soft", f"Median - {soft_desc}"),
        ]
        
        for key, title in blocks:
            worksheet.write(start_row, 0, title, title_format)
            start_row += 1
            
            worksheet.write(start_row, 0, "chain", header_format)
            worksheet.write(start_row, 1, "residue_pdb_resseq", header_format)
            worksheet.write(start_row, 2, "resname", header_format)
            start_row += 1
            
            rows = consensus.get(key.split("_")[0], {}).get(key.split("_")[1], [])
            for k in rows:
                chain, residue_pdb_resseq, resname = parse_residue_key(k) 
                worksheet.write(start_row, 0, chain)
                worksheet.write(start_row, 1, residue_pdb_resseq)
                worksheet.write(start_row, 2, resname)
                start_row += 1
            
            start_row += 1

        img_row, img_col = start_row, 0
        for _, imgpath in images:
            worksheet.insert_image(img_row, img_col, str(imgpath), {'x_scale': 1.0, 'y_scale': 1.0})
            if img_col == 0:
                img_col = 6
            else:
                img_col = 0
                img_row += 20

        writer.close()
        print(f"Global SUCCESS ✅ - Results saved to {output_path.name}")

    except Exception:
        writer.close()
        raise


def run_consensus_analysis(analysis_dir: str,
                           replicas: List[int],
                           chains: List[str],
                           mean_sheet_name: str,
                           median_sheet_name: str,
                           cols_required: List[str],
                           score_threshold: float,
                           output_filename_template: str) -> List[str]:
    """
    Main execution function.
    """
    analysis_path = Path(analysis_dir).resolve()
    
    print(f"\n>>> Starting Consensus Analysis (Threshold: {score_threshold:.3f})")
    print(f"Global INFO - Search Directory: {analysis_path}")

    # --- Step 1: Load Data ---
    replicas_mean: Dict[int, Optional[pd.DataFrame]] = {}
    replicas_median: Dict[int, Optional[pd.DataFrame]] = {}
    available_replicas: List[int] = []

    for r in replicas:
        path = analysis_path / REPLICA_FILENAME_TEMPLATE.format(repl=r)
        mean_df = load_replica_file(path, r, mean_sheet_name, cols_required)
        median_df = load_replica_file(path, r, median_sheet_name, cols_required)

        if mean_df is None and median_df is None:
            continue

        replicas_mean[r] = mean_df
        replicas_median[r] = median_df
        available_replicas.append(r)

    if not available_replicas:
        print("Global ERROR ❌ - No valid replicas found. Consensus analysis halted.")
        return []

    num_available = len(available_replicas)
    print(f"Global INFO - Found {num_available} valid replicas: {available_replicas}")

    # --- Step 2: Parallel Tables ---
    print("\nGlobal INFO - Calculating Parallel Tables and Filtering Keys")
    mean_parallel = build_parallel_sheet(replicas_mean, chains)
    median_parallel = build_parallel_sheet(replicas_median, chains)

    replicas_mean_sel: Dict[int, List[str]] = {}
    replicas_median_sel: Dict[int, List[str]] = {}

    for r in available_replicas:
        dfm, dfmed = replicas_mean.get(r), replicas_median.get(r)
        sel_mean, sel_median = [], []
        for chain in chains:
            sel_mean.extend(filter_residues_by_score(dfm, chain, score_threshold))
            sel_median.extend(filter_residues_by_score(dfmed, chain, score_threshold))

        replicas_mean_sel[r] = list(dict.fromkeys(sel_mean))
        replicas_median_sel[r] = list(dict.fromkeys(sel_median))
        print(f"Replica [{r}] INFO - Filtered: Mean={len(replicas_mean_sel[r])}, Median={len(replicas_median_sel[r])}")

    # --- Step 3: Consensus ---
    print("\nGlobal INFO - Computing Frequency Table and Consensus Levels")

    key_to_resseq_map: Dict[str, str] = {}
    for r in available_replicas:
        df_r = replicas_mean.get(r)
        if df_r is None:
            df_r = replicas_median.get(r)
        if df_r is not None and "residue_pdb_resseq" in df_r.columns:
            for _, row in df_r.iterrows():
                key_to_resseq_map[make_residue_key(row)] = str(row["residue_pdb_resseq"])

    all_mean_keys = [k for lst in replicas_mean_sel.values() for k in lst]
    mean_counts = Counter(all_mean_keys)
    all_median_keys = [k for lst in replicas_median_sel.values() for k in lst]
    median_counts = Counter(all_median_keys)
    
    union_keys = list(mean_counts.keys() | median_counts.keys())
    union_mean = order_union_preserve_first_appearance({r: replicas_mean[r] for r in available_replicas if replicas_mean.get(r) is not None}, chains)
    union_median = order_union_preserve_first_appearance({r: replicas_median[r] for r in available_replicas if replicas_median.get(r) is not None}, chains)
    
    ordered_keys = [k for k in union_mean + union_median if k in union_keys]
    freq_keys = list(dict.fromkeys(ordered_keys))

    freq_rows = []
    for k in freq_keys:
        chain_id, residue_pdb_resseq_raw, resname = parse_residue_key(k)
        freq_rows.append({
            "chain": chain_id, 
            "residue_pdb_resseq": key_to_resseq_map.get(k, residue_pdb_resseq_raw),
            "resname": resname,
            "mean_count": mean_counts[k], "median_count": median_counts[k], 
            "total_count": mean_counts[k] + median_counts[k]
        })

    freq_df = pd.DataFrame(freq_rows, columns=["chain", "residue_pdb_resseq", "resname", "mean_count", "median_count", "total_count"])

    # --- LOGIC CORRECTION APPLIED HERE ---
    # Soft threshold is ceil(N/2). 
    # Example N=5: ceil(2.5)=3. Soft range: [3, 4]. Hard: 5.
    soft_threshold_val = int(np.ceil(num_available / 2)) if num_available > 0 else 0

    print(f"Global INFO - Applying Consensus Logic for {num_available} replicas.")
    print(f"   > Hard Consensus: count == {num_available}")
    print(f"   > Soft Consensus: {soft_threshold_val} <= count < {num_available}")
    
    mean_hard_keys = [k for k, cnt in mean_counts.items() if cnt == num_available]
    mean_soft_keys = [k for k, cnt in mean_counts.items() if cnt >= soft_threshold_val and cnt < num_available]

    median_hard_keys = [k for k, cnt in median_counts.items() if cnt == num_available]
    median_soft_keys = [k for k, cnt in median_counts.items() if cnt >= soft_threshold_val and cnt < num_available]

    def order_consensus_list(keys: List[str], union_order: List[str], chains: List[str]) -> List[str]:
        ordered = []
        for ch in chains:
            for k in union_order:
                if k in keys and parse_residue_key(k)[0] == ch:
                    ordered.append(k)
        return ordered

    consensus_dict = {
        "mean": {"hard": order_consensus_list(mean_hard_keys, union_mean, chains),
                 "soft": order_consensus_list(mean_soft_keys, union_mean, chains)},
        "median": {"hard": order_consensus_list(median_hard_keys, union_median, chains),
                   "soft": order_consensus_list(median_soft_keys, union_median, chains)}
    }

    # --- Step 4: Plots and Save ---
    print("\nGlobal INFO - Generating Plots and Writing Final Excel Output")
    tmpdir = Path(tempfile.mkdtemp(prefix="consensus_imgs_"))
    images: List[Tuple[str, str]] = [] 
    
    try:
        def create_plots_for(metric_name: str, union_keys: List[str], replicas_sel: Dict[int, List[str]], replicas_dfs_metric: Dict[int, Optional[pd.DataFrame]]):
            valid_replicas_dfs = {r: df for r, df in replicas_dfs_metric.items() if df is not None}
            
            fig3 = plot_per_replica_totals(valid_replicas_dfs, chains, f"{metric_name.upper()} - Per Replica Totals")
            f3path = tmpdir / f"{metric_name}_totals.png"
            fig3.savefig(f3path, dpi=150); plt.close(fig3)
            images.append((f"{metric_name.upper()} Totals", str(f3path)))

            for ch in chains:
                chain_keys = [k for k in union_keys if parse_residue_key(k)[0] == ch]
                if not chain_keys: continue
                
                label_base = f"{metric_name.upper()} Chain {ch}"
                
                fig1 = plot_frequency_distribution(chain_keys, replicas_sel, f"{label_base} - Frequency Distribution")
                f1path = tmpdir / f"{metric_name}_chain{ch}_freq.png"
                fig1.savefig(f1path, dpi=150); plt.close(fig1)
                images.append((f"{label_base} Frequency", str(f1path)))

                fig2 = plot_overlap_matrix(chain_keys, replicas_sel, f"{label_base} - Overlap Matrix")
                f2path = tmpdir / f"{metric_name}_chain{ch}_overlap.png"
                fig2.savefig(f2path, dpi=150); plt.close(fig2)
                images.append((f"{label_base} Overlap", str(f2path)))

        create_plots_for("mean", union_mean, replicas_mean_sel, replicas_mean)
        create_plots_for("median", union_median, replicas_median_sel, replicas_median)
        
        summary_content = {
            "available_replicas": available_replicas, 
            "score_threshold": score_threshold,
            "freq_table": freq_df, 
            "consensus": consensus_dict,
            "soft_threshold_val": soft_threshold_val,
        }

        output_filename = output_filename_template.format(score_threshold=score_threshold)
        output_path = analysis_path / output_filename

        write_excel_output(output_path, mean_parallel, median_parallel, summary_content, images)

    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)
        print("Global INFO - Cleanup: Temporary image files removed.")

    return [str(output_path)]


if __name__ == "__main__":
    analysis_dir = SEARCH_ROOT_DIR
    score_threshold = SCORE_THRESHOLD_DEFAULT
    output_template = OUTPUT_XLSX_PATTERN
    chains_list = DEFAULT_CHAINS

    replicas_list = find_available_replicas(analysis_dir, REPLICA_FILENAME_TEMPLATE)
    print(f"Global INFO - Found the following replicas dynamically: {replicas_list}")

    generated_files = run_consensus_analysis(analysis_dir=analysis_dir,
                                             replicas=replicas_list,
                                             chains=chains_list,
                                             mean_sheet_name=SHEET_MEAN_DATA,
                                             median_sheet_name=SHEET_MEDIAN_DATA,
                                             cols_required=COLS_REQUIRED,
                                             score_threshold=score_threshold,
                                             output_filename_template=output_template)
    
    if generated_files:
        print("\nGlobal INFO - Consensus generation complete. Generated file:")
        for p in generated_files:
            print(p)
    else:
        print("\nGlobal ERROR ❌ - Consensus generation failed. No files generated.")